In [0]:
%pip install --trusted-host pypi.tuna.tsinghua.edu.cn --index-url https://pypi.tuna.tsinghua.edu.cn/simple/ "faiss-cpu>=1.9.0"
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from pathlib import Path
import os
import pickle
try:
    from threadpoolctl import threadpool_limits
    THREAD_CONTROL = True
except Exception:
    THREAD_CONTROL = False
try:
    import faiss
    _faiss_available = True
except ImportError:
    _faiss_available = False
from pyspark.sql.types import StructType, StructField, ArrayType, StringType
from pyspark.sql.functions import lit  
import json

In [0]:
class ItemToItemRetriever:
    def __init__(self, top_n=10, method="faiss", faiss_index_type="ivf_flat", nlist=16, nprobe=4, m_pq=32, nbits=8, use_gpu=False):
        """
        method: 'exact' (cosine similarity) or 'faiss'
        faiss_index_type: 'ivf_flat' (exact faiss indexes) or 'ivf_pq' (compressed faiss indexes)
        nlist: number of clusters used for cell-probe method. The feature space is partitioned into nlist cells.
        nprobe: number of clusters to explorer at search timeAt query time, a set of nprobe inverted lists is selected
        m_pq: number of subquantizers for PQ (only for 'ivf_pq')
        nbits: bits per code 
        use_gpu: if using faiss, whether to run on GPU
        For index choices, refer to
        https://github.com/facebookresearch/faiss/wiki/Faiss-indexes
        https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index
        """
        self.top_n = top_n
        self.method = method
        self.faiss_index_type = faiss_index_type
        self.nlist = nlist
        self.nprobe = nprobe 
        self.m_pq = m_pq 
        self.nbits = nbits 
        self.use_gpu = use_gpu

        self.item_id_to_index = {}
        self.index_to_item_id = {}

        self.sim_matrix = None  # for 'exact'
        self.faiss_index = None  # for 'faiss'
        self.item_vectors = None

    def fit(self, interactions: pd.DataFrame = None, item_vectors: np.ndarray = None, item_ids: list = None):
        """
        Fit the model using either:
        - interactions: DataFrame with ['user_id', 'item_id', 'interaction']
        - OR
        - item_vectors: ndarray (n_items x dim)
          and item_ids: list of item_id strings or ints matching row order
        """
        if interactions is not None:
            self._fit_from_interactions(interactions)
        elif item_vectors is not None and item_ids is not None:
            self._fit_from_embeddings(item_vectors, item_ids)
        else:
            raise ValueError("Must provide either `interactions` or (`item_vectors` and `item_ids`).")

        if self.method == "exact":
            self.sim_matrix = cosine_similarity(self.item_vectors)
        elif self.method == "faiss":
            if not _faiss_available:
                raise ImportError("FAISS is not installed. Try: pip install faiss-cpu")
            self._build_faiss_index()
        else:
            raise ValueError(f"Unknown method: {self.method}")

    def _get_item_index_map(self, item_ids: list):
        self.item_id_to_index = {id_: idx for idx, id_ in enumerate(item_ids)}
        self.index_to_item_id = {idx: id_ for id_, idx in self.item_id_to_index.items()}

    def _fit_from_interactions(self, interactions: pd.DataFrame):
        item_ids = interactions['item_id'].unique()
        user_ids = interactions['user_id'].unique()
        self._get_item_index_map(item_ids)
        user_id_to_index = {id_: idx for idx, id_ in enumerate(user_ids)}

        row = interactions['user_id'].map(user_id_to_index)
        col = interactions['item_id'].map(self.item_id_to_index)
        data = interactions['interaction']

        user_item_matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), len(item_ids)))

        self.item_vectors = np.ascontiguousarray(user_item_matrix.T.toarray(), dtype=np.float32)

    def _fit_from_embeddings(self, item_vectors: np.ndarray, item_ids: list):
        self.item_vectors = np.ascontiguousarray(item_vectors, dtype=np.float32)
        self._get_item_index_map(item_ids)

    def _build_faiss_index(self):
        # Normalize vectors for cosine similarity via inner product
        faiss.normalize_L2(self.item_vectors)
        d = self.item_vectors.shape[1]

        if self.faiss_index_type == "ivf_flat":
            quantizer = faiss.IndexFlatIP(d) # index based on inner product
            index = faiss.IndexIVFFlat(quantizer, d, min(self.nlist, len(self.item_vectors)), faiss.METRIC_INNER_PRODUCT)
        elif self.faiss_index_type == "ivf_pq":
            quantizer = faiss.IndexFlatIP(d)
            index = faiss.IndexIVFPQ(quantizer, d, min(self.nlist, len(self.item_vectors)), self.m_pq, self.nbits)
        else:
            raise ValueError(f"Unsupported FAISS index type: {self.faiss_index_type}")

        if self.use_gpu:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)

        # Train the index to find a suitable clustering
        index.train(self.item_vectors)
        index.add(self.item_vectors)
        index.nprobe = min(self.nprobe, len(self.item_vectors))

        self.faiss_index = index

    def get_similar_items(self, item_id, return_score=False):
        """
        Retrieve top-N similar items for a given item_id
        Returns: List of (item_id, similarity) or List of item_id
        """
        if item_id not in self.item_id_to_index.keys():
            raise ValueError(f"Item ID {item_id} not in index.")

        idx = self.item_id_to_index[item_id]

        if self.method == "exact":
            sims = self.sim_matrix[idx]
            top_indices = np.argsort(-sims)
            top_indices = top_indices[top_indices != idx][:self.top_n]
            if return_score:
                return [(self.index_to_item_id[i], float(sims[i])) for i in top_indices]
            else: 
                return [self.index_to_item_id[i] for i in top_indices]

        elif self.method == "faiss":
            query_vec = self.item_vectors[idx].reshape(1, -1)
            faiss.normalize_L2(query_vec)
            sims, indices = self.faiss_index.search(query_vec, self.top_n + 1)
            results = []
            for i, s in zip(indices[0], sims[0]):
                if i == -1 or i == idx:
                    continue  # skip self and invalid index
                if return_score:
                    results.append((self.index_to_item_id[i], float(s)))
                else:
                    results.append(self.index_to_item_id[i])
                if len(results) >= self.top_n:
                    break
    
            return results

    def get_similar_items_for_all(self, return_score=False):
        """
        Returns a dict: item_id -> list of (similar_item_id, similarity) or list of similar_item_id
        """
        return {item_id: self.get_similar_items(item_id, return_score) for item_id in self.item_id_to_index}

In [0]:
def retrieve_user_post_interaction_data(spark, db, table_names, run_date, last_n_interaction=100, weights={'click':1,'like':2,'share':5,'comment':7,'publish':10}, to_pandas=False): 
    sdf = spark.sql("""
                    WITH interaction_degrees AS (
                      SELECT user_id, 
                      post_id as item_id,
                      partition_date,
                      SUM(interaction_degree) as interaction
                      FROM (
                        SELECT
                          user_id,
                          post_id,
                          partition_date,
                          CASE
                            WHEN interaction = 'view' THEN {w1}
                            WHEN interaction = 'like' THEN {w2}
                            WHEN interaction = 'share' THEN {w3}
                            WHEN interaction = 'comment' THEN {w4}
                            WHEN interaction = 'publish' THEN {w5}
                            ELSE 0
                          END AS interaction_degree
                        FROM {db}.{user_post_interaction}
                        WHERE partition_date <= '{run_date}'
                      )
                      GROUP BY 1, 2, 3
                    ),
                    ranked_pos_samples AS (
                      SELECT
                        *,
                        ROW_NUMBER() OVER (
                          PARTITION BY user_id
                          ORDER BY partition_date DESC
                        ) AS rn
                      FROM interaction_degrees
                      WHERE interaction > 0
                    )            
                    SELECT user_id, item_id, interaction
                    FROM ranked_pos_samples
                    WHERE rn <= {last_n_interaction}
                    """.format(db = db,
                              user_post_interaction = table_names["user_post_interaction"],
                              run_date = run_date,
                              last_n_interaction = last_n_interaction,
                              w1 = weights['click'],
                              w2 = weights['like'],
                              w3 = weights['share'],
                              w4 = weights['comment'],
                              w5 = weights['publish'])
                  )
    return sdf.toPandas() if to_pandas else sdf

In [0]:
# ============================================================
# Inference functions
# ============================================================
def build_item_interactions(df, user_col="user_id", item_col="item_id", interaction_col="interaction"):
    """Return dict: user_id -> set(of positive item_ids)."""
    interactions = defaultdict(set)
    for _, row in df.iterrows():
        if row[interaction_col] > 0:
            interactions[row[user_col]].add(row[item_col])
    return interactions

def run_i2i(retriever, df, user_col="user_id", item_col="item_id", interaction_col="interaction", top_k=100, sim_item_indices=None):
    """
    For each positive (user, item) in test, query similar items for 'item' and return top_k of them.
    """
    data_inter = build_item_interactions(df, user_col, item_col, interaction_col)
    retrieved = dict.fromkeys(df[user_col].unique())

    for user_id, pos_items in data_inter.items():
        item_set = []
        if not pos_items:
            continue
        if sim_item_indices:
            # For each query item, if the precomputed similar item indices are given, directly fetch the results
            for item in pos_items:
                try:
                    item_set.extend(sim_item_indices[item])
                except Exception:
                    # item may be unseen in training/index
                    continue
        else:
            for item in pos_items:
                try:
                    item_set.extend(retriever.get_similar_items(item, return_score=False))
                except Exception:
                    # item may be unseen in training/index
                    continue
        retrieved[user_id] = list(set(item_set))[:top_k]

    return retrieved

def retrieved_dict_to_dataframe(spark, retrieved_dict, to_pandas=False):
    # Convert dict to list of tuples
    data = [(k, v) for k, v in retrieved_dict.items()]

    # define schema
    schema = StructType([
        StructField("user_id", StringType(), False),
        StructField("post_id", ArrayType(StringType()), False)
    ])
    
    sdf = spark.createDataFrame(data, schema=schema)
    return sdf.toPandas() if to_pandas else sdf

In [0]:
def select_active_users(spark, run_date, lookback_win, db, table_names, to_pandas=False):
    sdf = spark.sql("""
                    select distinct user_id 
                    from {db}.{user_attributes}
                    where community_visits > 0 
                    and partition_date between date_sub('{run_date}', {lookback_win}) and '{run_date}'
                    """.format(db = db,
                               user_attributes = table_names["user_attributes"],
                               run_date = run_date,
                               lookback_win = lookback_win)
    )
    return sdf.toPandas() if to_pandas else sdf

def filter_active_users(df, user_set_df):
    # keeps rows in df where user_id exists in user_set_df
    filtered_df = df.join(user_set_df.select("user_id"), on="user_id", how="left_semi")
    return filtered_df 

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
weights = model_config['interaction_weights']
last_n_interaction = model_config['last_n_interactions']
df = retrieve_user_post_interaction_data(spark, db, table_names, run_date, last_n_interaction, weights, to_pandas=True)

In [0]:
model_path = model_config['I2I_MODEL_PATH']
retriever = pickle.load(open(Path(model_path).joinpath('i2iCF_retriever.pkl'),'rb'))
sim_item_ind = pickle.load(open(Path(model_path).joinpath('i2iCF_similar_item_indices.pkl'),'rb'))

In [0]:
top_k = model_config['top_k_posts']
retrieved = run_i2i(retriever, df, user_col="user_id", item_col="item_id", interaction_col="interaction", top_k=top_k, sim_item_indices=sim_item_ind)

In [0]:
sdf = retrieved_dict_to_dataframe(spark, retrieved)

In [0]:
lookback_win = model_config['active_user_window']
user_set_sdf = select_active_users(spark, run_date, lookback_win, db, table_names)
sdf = filter_active_users(sdf, user_set_sdf)

In [0]:
sdf = sdf.withColumn("partition_date", lit(run_date))

In [0]:
# Write data to table
sdf.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(db+'.'+table_names['i2i_retrieval'])